# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://mlcommons.github.io/croissant/main/) library. The dataset is defined via a Croissant schema and provides ordered logistic regression outputs for adoption predictors of rangeland management knowledge in Northern Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata using `mlcroissant`. The metadata describes the dataset's record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("\nDataset Description:\n", metadata.description)


## 2. Data Overview
List available record sets, their names, and field `@id`s. The `@id` of each element is used for referencing according to Croissant schema best practices.

In [ ]:
# Explore available record sets and their fields
print("\nAvailable record sets (with '@id' and name):\n")

record_sets_info = []
for record_set in metadata.record_sets:
    print(f"Record set @id: {record_set.id}")
    print(f"  Name: {record_set.name}\n  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}, Name: {field.name}")
    print("")
    record_sets_info.append({
        'id': record_set.id,
        'name': record_set.name,
        'fields': [ {'id': f.id, 'name': f.name} for f in record_set.fields]
    })

if len(record_sets_info) == 0:
    print("No record sets are directly listed in top-level metadata. Attempting to infer record set ids from the dataset...")
    # Try to infer from available methods:
    print("mlcroissant will usually auto-detect available record_sets via dataset.metadata.record_sets.")
    # Since the metadata.record_sets is empty, we cannot provide concrete @id's. However, mlcroissant still supports iterating via available ids, e.g. from dataset.record_sets().
else:
    print("Record sets summary complete.")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

If no record sets are found in the metadata, use [`dataset.record_sets()`](https://mlcommons.github.io/croissant/reference/python/generated/mlcroissant.Dataset/#mlcroissant.Dataset.record_sets) to enumerate available record set `@id`s.

In [ ]:
# Retrieve all record set @id's known to this dataset
record_set_ids = dataset.record_sets()
print("Detected record set @id's:", record_set_ids)

# For illustration, select the first detected record set
if record_set_ids:
    chosen_record_set_id = record_set_ids[0]
    print(f"\nUsing record set @id: {chosen_record_set_id}")
else:
    raise RuntimeError("No record sets found in the Croissant schema.")

# Load the records for this record set
records = list(dataset.records(record_set=chosen_record_set_id))
df = pd.DataFrame(records)
print(f"Columns in '{chosen_record_set_id}':\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, e.g., filtering records, normalizing numeric fields, removing outliers, or grouping data. All operations reference record sets and fields by their `@id` according to the Croissant schema.

In [ ]:
# Let's select a numeric field automatically, or fall back on all available columns
import numpy as np

df_numeric = df.select_dtypes(include=[np.number])

if not df_numeric.empty:
    numeric_field_id = df_numeric.columns[0]  # pick the first numeric column by @id
else:
    # Try to convert viable columns to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    df_numeric = df.select_dtypes(include=[np.number])
    if not df_numeric.empty:
        numeric_field_id = df_numeric.columns[0]
    else:
        raise RuntimeError("No numeric fields available for EDA.")

print(f"Selected numeric field for filtering and normalization: {numeric_field_id}")

# Define a threshold, for demonstration use the median:
threshold = df_numeric[numeric_field_id].median()

# Filter records where value > threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Number of records with {numeric_field_id} > {threshold}: {len(filtered_df)}")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric values (Z-score normalization)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' sample:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a field if a categorical/text column exists
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = None
for col in categorical_cols:
    # Use the first non-id string column as group, if found
    if col.lower() not in ('id', '@id'):
        group_field = col
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable categorical field available for grouping.")

## 5. Visualization
Visualize the distribution of the numeric field, and if a grouping variable is available, show groupwise comparisons.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=20, kde=True, ax=ax, color='skyblue')
ax.set_title(f"Distribution of {numeric_field_id}")
ax.set_xlabel(numeric_field_id)
plt.show()

if group_field is not None:
    # Show boxplot by group
    plt.figure(figsize=(10,4))
    sns.boxplot(data=df, x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook loaded the FAIR² dataset using its Croissant schema, explored record sets and fields by their `@id`, extracted records using `mlcroissant`, and performed sample exploratory analysis and visualization. All entity references were made using their Croissant `@id`s to ensure unambiguous data selection and reproducibility.

Further analysis can build on this framework, using additional field or record set IDs as needed for advanced feature engineering or modeling tasks.

*Notebook generated by MLCommons FAIR² and `mlcroissant`.*